In [1]:
# (C) Martin Reißel

from sympy import *

from IPython.display import display, Math, Latex
from sympy.interactive import printing
printing.init_printing(use_latex='mathjax')
platex = lambda A: latex(A, mat_str='pmatrix', mat_delim='')

import pylab as pl

# Aufgabenstellung

Führen Sie für die Funktion

In [2]:
class DeferredVectorN(DeferredVector):
   def __getitem__(self, i):
        if i == -0:
            i = 0
        if i < 0:
            raise IndexError('DeferredVector index out of range')
        component_name = r'%s_%d' % (self.name, i)
        return Symbol(component_name)

x = DeferredVectorN('x')

x12 = (x[1], x[2])
f = Lambda(x12, (x[1] - x[2])**2 + (x[1] + x[2])**2 + x[1])

Math('f(x) = ' + latex(f(*x12)))

<IPython.core.display.Math object>

einen Schritt des Steepest-Descent-Verfahrens mit 

- exakter Liniensuche 

- Armijo Liniensuche für $\rho = \tau = \frac{1}{3}$
  und Startwert $\alpha = 1$

durch. Benutzen Sie den Startwert

In [3]:
rho = tau = Rational(1,3)
al0 = 1

x0 = Matrix([1,1])

Math( r'x = ' + platex(x0))

<IPython.core.display.Math object>

# Lösung

Bei Steepest-Descent gilt
$$
x_\text{neu} = x - \alpha f'(x),
$$
wobei $\alpha > 0$ dann mit Liniensuche bestimmt wird.

## Exakte Liniensuche

Bei exakter Liniensuche wird $\alpha$ bestimmt durch
$$
\varphi(\alpha) = \min_{s\geq0} \varphi(s),
\quad
\varphi(s) = f(x - s f'(x)).
$$
Mit

In [4]:
f1 = Lambda(x12, Matrix([f(*x12)]).jacobian([x[1],x[2]]))
Math(r"f'(x) = " + platex(f1(*x12).T) + r",\quad x = " + platex(x0))

<IPython.core.display.Math object>

folgt

In [5]:
s = symbols('s')
xs = x0 - s*f1(*x0).T

Math(r"x - sf'(x) = " + platex(xs))

<IPython.core.display.Math object>

und deshalb

In [6]:
phi = Lambda(s, f(*xs).simplify())

Math(r"\varphi(s) = f(x - sf'(x)) = " + platex(phi(s)))

<IPython.core.display.Math object>

Um $\varphi$ über $[0,\infty)$ zu minimieren betrachten wir alle lokalen
Extrema sowie $\varphi(0)$ und $\lim_{s\to\infty}\varphi(s)$.
Für die lokalen Extrema erhalten wir mit

In [7]:
phi1 = Lambda(s, phi(s).diff(s))
Math(r"\varphi'(s) = " + platex(phi1(s)) )

<IPython.core.display.Math object>

In [8]:
sex = solve(phi1(s), s)
pex = list(map(phi, sex))
Math(r"\hat{s} \in" + latex(sex) + r",\qquad \varphi(\hat{s}) \in" + latex(pex))

<IPython.core.display.Math object>

und für $s=0$ bzw. $s\to\infty$

In [9]:
p0 = phi(0)
pinf = phi(s).limit(s, oo)
Math(r"\varphi(0)=" + latex(p0) + r",\qquad \lim_{s\to\infty}\varphi(s) =" + latex(pinf))

<IPython.core.display.Math object>

In [10]:
ss = sex + [0]
pp = pl.array(map(phi, ss))

smin = ss[pp.argmin()]
if pinf < phi(smin):
    smin = oo

al = smin

also

In [11]:
Math(r'\alpha =' + latex(al))

<IPython.core.display.Math object>

und somit

In [12]:
xn = x0 - al * f1(*x0).T
Math(r"x_\text{neu} = x - \alpha f'(x) = " 
     + platex(x0) + "-" + latex(al) + platex(f1(*x0).T)
     + "=" + platex(xn))

<IPython.core.display.Math object>

und

In [13]:
Math(r'f(x) = {} \qquad f(x_\text{{neu}}) = {}'.format(f(*x0), f(*xn)))

<IPython.core.display.Math object>

## Armijo Liniensuche

Bei Armijo Liniensuche bestimmt man nicht das globale Minimimum
von $\varphi(s)$ für $s\geq 0$, man gibt sich damit zufrieden, wenn
man ein $\alpha \ge 0$ findet, für das
$$
\varphi(\alpha) \le \lambda(\alpha) = \varphi(0) + \rho \varphi'(0) \alpha
$$
gilt. Wegen $\varphi'(0) = - \|f'(x)\|_2^2$ und $\rho = \frac{1}{3}$ können wir das für $\alpha>0$ hinreichend nahe bei 0 immer erreichen.

Ein geeignetes $\alpha$ ermittelt man mit Backtracking. Wir starten
mit $\alpha = 1$ und testen, ob $\varphi(\alpha) \le \lambda(\alpha)$.
Trifft das zu, dann haben wir ein passendes $\alpha$ gefunden.
Wenn nicht, dann reduzieren wir $\alpha$ um den Faktor $\tau = \frac{1}{3}$ (also $\alpha_\text{neu} = \tau\alpha$) und wiederholen
das Ganze.

Aus dem ersten Teil wissen wir, dass

In [14]:
Math(r"\varphi(s) = f(x - sf'(x)) = " + platex(phi(s)))

<IPython.core.display.Math object>

Mit $\rho = \frac{1}{3}$ und

In [15]:
Math(r"\varphi'(s) = " + platex(phi1(s)) )

<IPython.core.display.Math object>

erhalten wir

In [16]:
lamb = Lambda(s, phi(0) + rho * phi1(0) * s)
Math(r"\lambda(s) = " + platex(lamb(s)) )

<IPython.core.display.Math object>

Führen wir jetzt mit Startwert $\alpha=1$ und $\tau = \frac{1}{3}$
das Backtracking durch, so erhalten wir folgende Werte

In [17]:
al = al0

while(phi(al) > lamb(al)):
    display(Math(r'\alpha = {} \qquad   \varphi(\alpha) = {}  \qquad \lambda(\alpha) = {}'.format(al, phi(al), lamb(al))))
    al = tau * al

display(Math(r'\alpha = {} \qquad   \varphi(\alpha) = {}  \qquad \lambda(\alpha) = {}'.format(al, phi(al), lamb(al))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

und somit

In [18]:
xn = x0 - al * f1(*x0).T
Math(r"x_\text{neu} = x - \alpha f'(x) = " 
     + platex(x0) + "-" + latex(al) + platex(f1(*x0).T)
     + "=" + platex(xn))

<IPython.core.display.Math object>

und

In [19]:
Math(r'f(x) = {} \qquad f(x_\text{{neu}}) = {}'.format(f(*x0), f(*xn)))

<IPython.core.display.Math object>